# DuckLake — Streaming-generate-and-query demo

Attach to the existing DuckLake catalog, create a table by streaming `N` synthetic rows
through the benchmark's `StreamingGenerator`, then query it interactively.

Memory is bounded by `chunk_size` — you can set `ROW_COUNT` to tens of millions on a
16 GB host without OOM. The benchmark's defaults (event_date partition col, dict-encoded
varchar, ~30 partitions) apply.

Edit the **Parameters** cell, run all cells, then iterate on the **Query** cell.


In [1]:
import sys
from pathlib import Path

from loguru import logger

logger.remove()
logger.add(sys.stderr, level="INFO")

_BENCH_DIR = Path.cwd()
if _BENCH_DIR.name != "benchmark":
    _BENCH_DIR = _BENCH_DIR / "benchmark"
    print(f"Changing benchmark directory to {_BENCH_DIR}")
if str(_BENCH_DIR) not in sys.path:
    sys.path.insert(0, str(_BENCH_DIR))

Changing benchmark directory to /Users/graziano/GitHub/poor-man-lakehouse/notebooks/benchmark


## Parameters

Tune these and re-run. `ROW_COUNT` is the headline knob.


In [2]:
from benchmark.helpers import (
    GeneratorSpec,
    StreamingGenerator,
    load_config,
    measure_time_and_memory,
)
from benchmark.helpers.engines.ducklake_engine import DuckLakeEngine

config = load_config(_BENCH_DIR / "config.yaml")
config.name

'DuckLake v1.0 vs Delta-rs + Polars'

In [3]:
ROW_COUNT = 10_000  # Number of rows to generate.
TABLE_NAME = "demo_table"  # Name of the DuckLake table to create.
STORAGE_MODE = "local"  # 'local' or 's3' (must be in config.storage_modes).
CHUNK_SIZE = config.batch_size  # Streaming batch size; defaults to YAML batch_size.
SEED = config.schema.seed  # Same seed as the benchmark for reproducibility.

assert STORAGE_MODE in config.storage_modes, f"Unknown storage mode {STORAGE_MODE}. Available: {config.storage_modes}"
f"Will create {TABLE_NAME} with {ROW_COUNT:,} rows in {STORAGE_MODE} storage"

'Will create demo_table with 10,000 rows in local storage'

## Attach to the existing DuckLake catalog

Reuses the benchmark's engine setup — installs extensions, creates secrets, attaches the
Postgres-backed catalog at the configured `data_path`. If the catalog DB doesn't exist yet,
the engine creates it.


In [4]:
engine = DuckLakeEngine()
engine.setup(config, STORAGE_MODE)
con = engine._con  # Use the engine's DuckDB connection for queries.
catalog = engine._catalog_name
fq = f"{catalog}.main.{TABLE_NAME}"
fq

2026-05-01 21:54:05.851 | INFO     | benchmark.helpers.engines.ducklake_engine:setup:120 - DuckLake engine setup complete (storage=local, data_path=/Users/graziano/GitHub/poor-man-lakehouse/notebooks/benchmark/data/ducklake/, pg_baseline=8867.1 KB)


'bench_ducklake_local.main.demo_table'

## Stream-generate and write

`StreamingGenerator` produces partition-sorted record batches; the engine consumes them via
a registered `pa.RecordBatchReader` and writes one DuckLake snapshot. No full materialization
of the dataset happens in Python.


In [5]:
gen = StreamingGenerator(
    GeneratorSpec(
        schema_config=config.schema,
        total_rows=ROW_COUNT,
        chunk_size=CHUNK_SIZE,
        seed=SEED,
    )
)
with measure_time_and_memory() as t:
    engine.write_overwrite(TABLE_NAME, gen.arrow_reader(), gen.schema)
timing = t[0]
print(
    f"Wrote {ROW_COUNT:,} rows in {timing.wall_time_seconds:.2f}s | "
    f"peak RSS {timing.peak_rss_mb:.0f} MB (delta {timing.delta_rss_mb:.0f} MB)"
)

Wrote 10,000 rows in 1.39s | peak RSS 261 MB (delta 97 MB)


## Schema and a sanity-check row count


In [6]:
schema_df = con.execute(f"DESCRIBE {fq}").pl()
schema_df

column_name,column_type,null,key,default,extra
str,str,str,str,str,str
"""id""","""BIGINT""","""YES""",null,"""NULL""",null
"""event_date""","""DATE""","""YES""",null,"""NULL""",null
"""int8_col""","""TINYINT""","""YES""",null,"""NULL""",null
"""int16_col""","""SMALLINT""","""YES""",null,"""NULL""",null
"""int32_col""","""INTEGER""","""YES""",null,"""NULL""",null
…,…,…,…,…,…
"""text_col""","""VARCHAR""","""YES""",null,"""NULL""",null
"""bool_col""","""BOOLEAN""","""YES""",null,"""NULL""",null
"""list_col""","""INTEGER[]""","""YES""",null,null,null


In [7]:
con.execute(f"SELECT COUNT(*) AS n, MIN(event_date) AS min_d, MAX(event_date) AS max_d FROM {fq}").pl()

n,min_d,max_d
i64,date,date
10000,2024-01-01,2024-01-30


## Query

Edit the SQL below and re-run. Wrapped in `measure_time_and_memory()` so you see wall time
and RSS for each query. Result is returned as a Polars DataFrame for easy slicing.


In [8]:
query = f"""
SELECT varchar_col,
       COUNT(*) AS cnt,
       SUM(int64_col) AS sum_val,
       AVG(float64_col) AS avg_val,
       MIN(event_date) AS min_date,
       MAX(event_date) AS max_date
FROM {fq}
WHERE event_date BETWEEN DATE '2024-01-10' AND DATE '2024-01-20'
GROUP BY varchar_col
ORDER BY cnt DESC
LIMIT 20
"""

with measure_time_and_memory() as t:
    result = con.execute(query).pl()
timing = t[0]
print(
    f"{result.height:,} rows in {timing.wall_time_seconds:.3f}s | "
    f"peak RSS {timing.peak_rss_mb:.0f} MB (delta {timing.delta_rss_mb:.0f} MB)"
)
result

20 rows in 0.031s | peak RSS 139 MB (delta 0 MB)


varchar_col,cnt,sum_val,avg_val,min_date,max_date
str,i64,"decimal[38,0]",f64,date,date
"""value_973""",11,-27342115746014120432,4.6440e13,2024-01-10,2024-01-20
"""value_746""",10,-19733138429620225528,-1.7911e14,2024-01-10,2024-01-20
"""value_540""",10,3097861329012073748,-2.1593e14,2024-01-11,2024-01-20
"""value_030""",9,-12145439426680215913,-3.0180e14,2024-01-12,2024-01-18
"""value_198""",9,14316339069882708585,-1.2177e14,2024-01-10,2024-01-20
…,…,…,…,…,…
"""value_243""",8,6806643308714807437,-3.3140e14,2024-01-10,2024-01-20
"""value_054""",8,-36109370310727947471,-3.7586e14,2024-01-11,2024-01-20
"""value_366""",8,3171123443060430248,-2.0140e14,2024-01-11,2024-01-19


## EXPLAIN ANALYZE (optional)

Useful for the talk: shows partition pruning and predicate pushdown in the plan.


In [9]:
plan_rows = con.execute(f"EXPLAIN ANALYZE {query}").fetchall()
print("\n".join(row[1] for row in plan_rows))

┌─────────────────────────────────────┐
│┌───────────────────────────────────┐│
││    Query Profiling Information    ││
│└───────────────────────────────────┘│
└─────────────────────────────────────┘
EXPLAIN ANALYZE  SELECT varchar_col,        COUNT(*) AS cnt,        SUM(int64_col) AS sum_val,        AVG(float64_col) AS avg_val,        MIN(event_date) AS min_date,        MAX(event_date) AS max_date FROM bench_ducklake_local.main.demo_table WHERE event_date BETWEEN DATE '2024-01-10' AND DATE '2024-01-20' GROUP BY varchar_col ORDER BY cnt DESC LIMIT 20 
┌────────────────────────────────────────────────┐
│┌──────────────────────────────────────────────┐│
││              Total Time: 0.0188s             ││
│└──────────────────────────────────────────────┘│
└────────────────────────────────────────────────┘
┌───────────────────────────┐
│           QUERY           │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│      EXPLAIN_ANALYZE      │
│    ────────────────────   │
│      

## Cleanup (optional)

Leaves the table on disk by default so you can keep iterating. Uncomment to drop it.


In [ ]:
# engine.teardown(TABLE_NAME)
# engine.close()